# Executor Memory Sizing and Overhead

**Module**: 06_memory_and_storage  
**Notebook**: 02_executor_memory.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Sizing spark.executor.memory and spark.executor.memoryOverhead, and Docker container memory limits.
- **Why does this topic exist?**: Incorrect sizing causes Linux kernel OOM kills (Exit Code 137).
- **What problem does it solve?**: Container killed because off-heap PySpark memory was ignored.
- **Where is it used in real-world Data Engineering?**: Production container resource requests in Kubernetes and Docker Compose.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Essential |
| **Estimated effort** | 2–3 h |
| **Prerequisites** | Spark memory architecture, container basics |
| **Required depth** | Working knowledge: can size executor memory and overhead for a container |
| **Data Engineer** | Set executor memory, overhead, and container limits consistently. |
| **Senior Data Engineer** | Diagnose container kills and tune memory without wasting cluster capacity. |
| **Spark specialist** | Reason about JVM, Python, off-heap, and OS memory together. |
| **Next topic** | `06_memory_and_storage/03_storage_and_execution_memory.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

Executor sizing is a budget across four layers: JVM heap, JVM overhead, application-native memory (especially
Python), and the operating system. A container limit that covers only `spark.executor.memory` will eventually kill
the executor. The goal is not the largest heap; it is a predictable total that leaves headroom for spikes.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Executor heap** | `spark.executor.memory` | Java heap for tasks, cache, and Spark internals |
| **Memory overhead** | `spark.executor.memoryOverhead` | Native/JVM/Python overhead outside the heap |
| **Container limit** | Total pod/container memory | Must cover heap plus overhead plus OS reserve |
| **PySpark memory** | `spark.executor.pyspark.memory` | Optional limit for Python worker processes |
| **Off-heap** | `spark.memory.offHeap.size` | Explicit native memory allocation |
| **GC** | Garbage collection | Large heaps can increase pause time |

### How it works, step by step

1. The driver requests an executor with a heap size and overhead.
2. The cluster manager creates a container with at least that total.
3. The JVM starts with the configured heap and reserves native memory.
4. Python workers and off-heap buffers allocate outside the heap.
5. If the container total exceeds its limit, the OS or orchestrator kills the process.
6. Spark retries or replaces the executor, but the job may fail repeatedly if the sizing is wrong.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Docker Container vs Spark Heap

`Container Limit >= spark.executor.memory + spark.executor.memoryOverhead + OS Buffer`

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Executor_Memory_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
overhead_factor = 0.10 # default 10%
exec_mem_mb = 2048 # 2 GB
calculated_overhead = max(384, int(exec_mem_mb * overhead_factor))
print(f"Configured Heap:     {exec_mem_mb} MB")
print(f"Calculated Overhead: {calculated_overhead} MB")
print(f"Total Process RAM:   {exec_mem_mb + calculated_overhead} MB")

---

## Experiment: Verifying Worker Capacity

- **1. What are we testing?**: Check worker capacity vs executor size in our cluster.
- **2. What do we expect?**: Worker has 4GB; Executor has 2GB -> 1-2 executors per worker.

In [ ]:
print('Worker memory from env: 4g, Executor memory: 2g')

**3. What happened?**: Matched Docker Compose configuration.  
**4. Why did it happen?**: Configured in .env.  
**5. Practical Engineering Lesson**: Ensure total executor requests do not exceed worker host capacity.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Kubernetes executor eviction

- **Industry**: SaaS platform
- **Scenario**: Spark executors on Kubernetes are evicted with OOMKilled during large shuffles.
- **Problem**: The pod memory limit equals the heap size, leaving no room for overhead or Python workers.
- **Why this concept is relevant**: Sizing the pod limit as heap plus overhead plus a reserve prevents the kernel from killing the process.
- **Implementation idea**: Compare pod memory usage with heap usage and set requests/limits explicitly.
- **Expected result**: Stable executors and fewer retries.
- **Alternative**: Use larger nodes with more headroom.
- **Trade-offs**: Larger nodes reduce eviction frequency but increase cost and failure blast radius.

### Use Case 2: Python UDF memory spike

- **Industry**: Marketing analytics
- **Scenario**: A pandas UDF job runs fine on small data but fails on the full dataset.
- **Problem**: Each Python worker loads a partition into memory, and that memory is outside the JVM heap.
- **Why this concept is relevant**: Accounting for `spark.executor.pyspark.memory` and the container limit explains the failure.
- **Implementation idea**: Inspect executor logs for Python worker kills and measure per-partition memory.
- **Expected result**: A sizing and partitioning change that keeps Python memory bounded.
- **Alternative**: Rewrite the UDF as native Spark expressions.
- **Trade-offs**: Native expressions avoid Python memory entirely but may not express complex logic.

---

## Business Value

- Fewer failed jobs and retries.
- Lower cloud cost from right-sized containers.
- Predictable performance during peak loads.
- Better collaboration between data and platform teams.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Always size the container above heap plus overhead.
- Account for Python workers and off-heap buffers explicitly.
- Avoid very large heaps when GC pauses dominate; prefer more moderate executors.
- Use memory requests and limits that match the Spark configuration.
- Test with the largest expected partition, not the average.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Calculate a safe container budget

**Steps**

1. Choose a heap size and compute the default overhead.
2. Add a reserve for Python workers and OS page cache.
3. Run a shuffle and a Python UDF under the resulting limit and watch container memory.

**What to observe**: The total container usage stays below the limit with headroom for spikes.

**Healthy result**: No OOMKilled events and stable executor count.

**Problematic result**: Memory usage approaches the limit during shuffle or UDF execution.

**Engineering takeaway**: Size the container, not just the heap.

---

## Trade-offs and Alternatives

### Advantages

- Explicit control
- Works across cluster managers
- Prevents most container kills

### Limitations

- Easy to misconfigure
- Python memory is hard to predict
- Large heaps increase GC pauses

### When to use it

- Kubernetes deployments
- YARN queues
- Standalone clusters

### When not to use it

- Heap equal to container limit
- Ignoring overhead
- Sizing from a small sample

### Alternatives inside Spark

- Executor memory
- Memory overhead
- PySpark memory
- Off-heap

### Alternatives outside Spark

- Kubernetes resource limits
- YARN memory settings
- JVM GC flags

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Calculate Overhead for 8GB
**Problem Statement**: Compute overhead for 8GB heap.  
**Input / Context**: 8192 MB

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Exit Code 137 Meaning
**Problem Statement**: Explain exit code 137.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): PySpark Overhead
**Problem Statement**: Why PySpark needs higher overhead.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### Executors keep dying under load

**Scenario**: Executors are replaced every few minutes during a large join.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check for OOMKilled or exit code 137 in the orchestrator events.
2. Compare container memory usage with heap and overhead settings.
3. Look for Python workers or off-heap allocations.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Calculate Overhead for 8GB):
```python
print(max(384, int(8192 * 0.10)))
```

#### Solution 2 (Exit Code 137 Meaning):
```python
print('Exit Code 137 = 128 + 9 (SIGKILL) = Container OOMKilled by Linux kernel')
```

#### Solution 3 (PySpark Overhead):
```python
print('PySpark spawns Python worker daemons outside the JVM heap.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Overhead**: Off-heap buffer for PySpark, Netty, Metaspace.
- **Docker Buffer**: Always provide a safety margin over heap.

### Essential Technical Terms
- `memoryOverhead`: Off-heap process allocation.
- `OOMKilled`: Kernel termination of process exceeding cgroup limit.

### Common Engineering Mistakes
- **Setting Docker limit equal to JVM heap**: Results in immediate container kill.

### Interview Questions on This Topic
- **Q: What causes Exit Code 137 in Spark?**
  - *A*: The Linux kernel OOM killer terminating the container because combined memory exceeded Docker memory limits.


### More Interview Questions

- **Q: What does `spark.executor.memoryOverhead` cover?**
  - *A*: Memory outside the JVM heap used by the JVM itself, native libraries, Python workers, and other process overhead; it must be included in the container limit.
- **Q: Why not simply give every executor a very large heap?**
  - *A*: Large heaps can increase GC pause time and the blast radius of a failure, and they may waste resources when partitions are small.

### What to Study Next
Open 03_storage_and_execution_memory.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')